---
title: "Convolutional Neural Networks (Part 1)"
short_title: Part 1
subject: DEEP LEARNING
---

## Introduction

Fully-connected neural networks are general and versatile. 
But its unconstrained nature make it difficult to train and generalize, especially for structured datasets. In particular, we consider image data
that can be represented as a grid structure such that nearby elements have strong local dependency. Common 
examples are images, sound, or similar sequential data with spatial orientation as key attribute.

In this chapter, we introduce the **convolution operation**, which can be thought of as a filter that is applied spatially in a homogenous manner. Stacking convolutional layers allow the network to learn hierarchical patterns that generalize well to test data. 
We will apply this architecture to image and text classification.

```{figure} ./img/03-lenet.png
---
align: center
name: 03-lenet
---
**LeNet-5 architecture** (1989). [**Source**](https://www.d2l.ai/chapter_convolutional-neural-networks/lenet.html)
```

In [ ]:
%matplotlib inline
%config InlineBackend.figure_format = "svg"
from tqdm import tqdm
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings("ignore", category=UserWarning)

import torch
import torch.nn as nn
import random
import numpy as np
import pandas as pd

RANDOM_SEED = 0
random.seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)
torch.manual_seed(RANDOM_SEED)

def get_device(): 
    return (
        torch.device("cuda:0") if torch.cuda.is_available() else (
            torch.device("mps") if torch.mps.is_available() else 
                torch.device("cpu")
        )
    )

DEVICE = get_device()
print(f"Using device: {DEVICE}")

In [ ]:
def savefig(filename: str):
    plt.savefig(f"./plots/{filename}.svg", bbox_inches="tight")
    plt.close("all");

def directive(filename, caption_topic="", caption=""):
    if caption_topic:
        fig_caption = f"**{caption_topic}.** {caption}"
    else:
        fig_caption = caption

    print(f"""
:::{{figure}} ./plots/{filename}.svg
---
name: {filename}
width: 100%
align: center
---
{fig_caption}
:::
    """)

# remove this! having caption args useful in copilot

## Convolution operation

Consider classifying images using a linear model. Flattening the image into a vector 
and feeding it into a fully-connected network is not the best approach.
The large flattened input vector requires a very large weight matrix.
Moreover, it does not consider local spatial correlation of image pixels ({numref}`cat-conv`)
(e.g. applying a fixed permutation to input data results in an equivalent network).
This motivates only mixing nearby pixels in a linear combination resulting in a very sparse banded weight matrix ({numref}`toeplitz`).

<br>

```{figure} ../../../img/cat.png
---
width: 400px
align: center
name: cat-conv
---
Nearby pixels combine to form meaningful features of an image. [Source](https://www.nationalgeographic.com/animals/mammals/facts/domestic-cat)
```

Let $\boldsymbol{\mathsf X}$ be an $n \times n$ input image and $\boldsymbol{\mathsf{S}}$ be the $m \times m$ output feature map. The banded weight matrix reduces the nonzero entries of the weight matrix from $m^2 n^2$ to $m^2{k}^2$ where a local region of $k \times k$ pixels in the input image are mixed. If the feature detector is translationally invariant across the image,
then the weights in each band are **shared**. This further reduces the number of weights to $O(k^2).$
The resulting linear operation is called a **convolution** in two spatial dimensions:


```{math}
\boldsymbol{\mathsf{S}}_{ij} = (\boldsymbol{\mathsf X} \circledast \boldsymbol{\mathsf{K}})_{ij} = \sum_{x = 0}^{{k}-1} \sum_{y=0}^{{k}-1} {\boldsymbol{\mathsf X}}_{i + x, j + y} \, {\boldsymbol{\mathsf{K}}}_{xy}.
```


Observe that spatial ordering of the pixels in the input $\boldsymbol{\mathsf X}$ is preserved in the output $\boldsymbol{\mathsf{S}}.$ This is nice since we want spatial information and orientation across a stack of convolution operations to be passed down into the final output.

```{figure} ../../../img/conv-cat-99.png
---
width: 30em
align: center
name: toeplitz
---
Banded Toeplitz matrix for classifying cat images [[Source]](https://deepimaging.github.io/lectures/lecture_9_intro_to_CNN's.pdf). The horizontal vectors contain the same pixel values. Note that there can be multiple bands for a 2D kernel. See this [SO answer](https://stackoverflow.com/a/44039201/1091950).
```

<br>

**Remark.** It can be shown that convolution has [translation equivariance](https://en.wikipedia.org/wiki/Equivariant_map) due to weight sharing. 

```{figure} ../../../img/nn/03-cnn-weight-sharing.png
---
width: 600px
align: center
name: 03-cnn-weight-sharing
---
Having local neural connectivity and weight sharing (both kernel and bias weights) characterizes the convolution. [Source](https://maurice-weiler.gitlab.io/blog_post/cnn-book_2_conventional_cnns/)
```

---